# Capítulo 11 — Detecção de Anomalias: Encontrando o que Foge do Normal

Códigos do capítulo. Execute as células em ordem; cada bloco é independente dos demais.


## 11.4.3 Regra do IQR (boxplot)

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
x = np.concatenate([rng.normal(100, 10, 200), [180, 195, 20]])   # 3 anomalias plantadas

z = (x - x.mean()) / x.std()
mad = np.median(np.abs(x - np.median(x)))
zmod = 0.6745 * (x - np.median(x)) / mad
q1, q3 = np.percentile(x, [25, 75])
iqr = q3 - q1

print("Z-score  |z|>3     :", np.where(np.abs(z) > 3)[0])
print("MAD      |z|>3.5   :", np.where(np.abs(zmod) > 3.5)[0])
print("IQR  (1,5 x IQR)   :", np.where((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr))[0])

## 11.4.4 Distância de Mahalanobis

In [ ]:
import numpy as np
from sklearn.covariance import EmpiricalCovariance, MinCovDet

rng = np.random.default_rng(0)
altura = rng.normal(170, 8, 300)
peso = 0.9 * (altura - 100) + rng.normal(0, 5, 300)
X = np.column_stack([altura, peso])
X = np.vstack([X, [190, 50]])                      # combinação atípica

robusto = MinCovDet(random_state=0).fit(X)         # estimador robusto de covariância
d2 = robusto.mahalanobis(X)                        # distância ao quadrado
print("Índices mais suspeitos:", np.argsort(d2)[-3:][::-1], " (o último é", len(X) - 1, ")")

## 11.5.2 Como funciona

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(0)
normais = rng.normal(0, 1, size=(500, 2))
anomalias = rng.uniform(-6, 6, size=(15, 2))
X = np.vstack([normais, anomalias])

iso = IsolationForest(n_estimators=200, contamination=0.03, random_state=0).fit(X)
pred = iso.predict(X)                 # +1 normal, -1 anomalia
escore = -iso.score_samples(X)        # quanto maior, mais anômalo

print("Anomalias apontadas:", (pred == -1).sum())
print("Acertos entre as 15 plantadas:", (pred[-15:] == -1).sum())

## 11.6.1 O problema da densidade variável

In [ ]:
import numpy as np
from sklearn.neighbors import LocalOutlierFactor

rng = np.random.default_rng(0)
denso   = rng.normal([0, 0], 0.3, size=(300, 2))
esparso = rng.normal([5, 5], 1.5, size=(100, 2))
suspeito = np.array([[1.2, 1.2]])                  # perto do denso, mas fora dele
X = np.vstack([denso, esparso, suspeito])

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
pred = lof.fit_predict(X)
escore = -lof.negative_outlier_factor_
print("Escore LOF do ponto suspeito:", round(escore[-1], 2))
print("Escore LOF mediano do resto :", round(float(np.median(escore[:-1])), 2))

## 11.7 One-Class SVM

In [ ]:
import numpy as np
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
treino = rng.normal(0, 1, size=(400, 2))              # só dados normais
teste = np.vstack([rng.normal(0, 1, (50, 2)),         # normais
                   rng.uniform(-6, 6, (10, 2))])      # possíveis anomalias

esc = StandardScaler().fit(treino)
modelo = OneClassSVM(kernel="rbf", gamma=0.3, nu=0.05).fit(esc.transform(treino))
pred = modelo.predict(esc.transform(teste))
print("Marcados como anomalia:", (pred == -1).sum(), "de", len(teste))

## 11.8.1 PCA para anomalias

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

rng = np.random.default_rng(0)
base = rng.normal(size=(500, 3))
ruido = 0.1 * rng.normal(size=(500, 10))
X = base @ rng.normal(size=(3, 10)) + ruido    # dados em ~3 dimensões
X[0] += 5 * rng.normal(size=10)                # ponto que quebra a estrutura

pca = PCA(n_components=3).fit(X)
erro = np.sum((X - pca.inverse_transform(pca.transform(X))) ** 2, axis=1)
print("Ponto mais anômalo:", erro.argmax())
print("Erro dele:", erro.max().round(1), "| erro mediano:", np.median(erro).round(2))

## 11.11 Como avaliar?

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

rng = np.random.default_rng(0)
X = np.vstack([rng.normal(0, 1, (1000, 4)), rng.normal(3, 1.5, (20, 4))])
y = np.r_[np.zeros(1000), np.ones(20)]                    # só para avaliar!

escore = -IsolationForest(random_state=0).fit(X).score_samples(X)
print("AUC-ROC:", round(roc_auc_score(y, escore), 3))
print("AUPRC  :", round(average_precision_score(y, escore), 3))
print("Linha de base AUPRC (acaso):", round(y.mean(), 3))

## Resumo do capítulo